# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Môi trường: notebook này chạy trên GPU host (Colab T4 hoặc Kaggle T4).
# Model, huấn luyện và suy luận chạy ở đây; máy local chỉ sửa mã và commit.
import os, socket, subprocess, sys
from pathlib import Path

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
KAGGLE_SLUG = 'lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes'

ON_KAGGLE = Path('/kaggle/working').is_dir()
ON_COLAB = (not ON_KAGGLE) and Path('/content').is_dir()
WORK = Path('/kaggle/working') if ON_KAGGLE else (Path('/content') if ON_COLAB else Path.cwd())
WORK.mkdir(parents=True, exist_ok=True)
print('platform:', 'kaggle' if ON_KAGGLE else ('colab' if ON_COLAB else 'local'), '| work dir:', WORK)


def _dns(host):
    try:
        socket.gethostbyname(host)
        return True
    except Exception:
        return False


print('internet -> pypi:', _dns('pypi.org'), '| huggingface:', _dns('huggingface.co'))
_probe = subprocess.run([sys.executable, '-c',
    "import torch;print(torch.__version__, torch.cuda.is_available(),"
    "torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')"],
    capture_output=True, text=True)
print('torch:', (_probe.stdout or _probe.stderr).strip()[-200:])
assert 'True' in (_probe.stdout or ''), 'Cần GPU: Colab -> Runtime/Change runtime type/T4 GPU; Kaggle -> Accelerator/T4 GPU'
assert _dns('huggingface.co'), 'Cần Internet để tải Qwen3-VL-4B-Instruct'


In [ ]:
# Lấy đúng mã hiện tại từ repo cá nhân (đã push lên GitHub), rồi cài đúng phiên bản đã thẩm định.
import os, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Yuhnguyn/K4-Track-4-Day10-VisionLanguageModel.git'
REPO = WORK / 'day10-lab'
if not (REPO / 'scripts' / 'score.py').is_file():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('repo:', Path.cwd())
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip())

!pip -q install -r requirements-colab.txt


## 1. Dữ liệu: tải và chia split

In [ ]:
# Nguồn dữ liệu: ưu tiên dataset Kaggle đã mount, sau đó Kaggle Secrets/CLI, cuối cùng ZIP công khai.
# Không khoá nào được ghi vào notebook hay commit.
import os, subprocess, sys, zipfile
from pathlib import Path

try:  # Chỉ có trên Colab; trên Kaggle sẽ ImportError và bỏ qua an toàn.
    from google.colab import userdata
    for _key in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        _value = userdata.get(_key)
        if _value:
            os.environ.setdefault(_key, _value)
except Exception:
    pass

ARGS = None
_candidates = []
if Path('/kaggle/input').is_dir():
    for _top in sorted(Path('/kaggle/input').iterdir()):
        if (_top / 'images').is_dir() and (_top / 'labels-YOLO').is_dir():
            _candidates.append(_top)
        _candidates += [p.parent for p in _top.rglob('labels-YOLO') if (p.parent / 'images').is_dir()]
if _candidates:
    ARGS = ['--source', str(_candidates[0])]
elif os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
    ARGS = ['--kaggle']
else:
    import requests
    _zip = Path('work/dataset.zip')
    _zip.parent.mkdir(parents=True, exist_ok=True)
    if not zipfile.is_zipfile(_zip):
        _response = requests.get(f'https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_SLUG}',
                                 stream=True, timeout=(30, 300))
        _response.raise_for_status()
        with _zip.open('wb') as _handle:
            for _chunk in _response.iter_content(1 << 20):
                _handle.write(_chunk)
        assert zipfile.is_zipfile(_zip), 'Tải công khai không trả ZIP: dùng Secrets hoặc --zip thủ công'
    ARGS = ['--zip', str(_zip)]

print('prepare_data.py', ARGS)
subprocess.run([sys.executable, 'scripts/prepare_data.py', *ARGS], check=True)


In [ ]:
# CP1: thống kê split, kiểm tra session và vẽ nhãn trên 6 ảnh train.
from student_data import explore
from IPython.display import display, Image as NotebookImage
data_summary = explore()
display(NotebookImage(filename='report/checkpoints/01-train-labels.png'))

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# CP2: nhãn JSON 0–1000, prompt và kiểm chứng quy ngược pixel.
from student_labels import PROMPT, yolo_to_box, objects_to_answer, verify_coordinates
coordinate_summary = verify_coordinates()
print(PROMPT)
display(NotebookImage(filename='report/checkpoints/02-coordinate-roundtrip.png'))

## 3. E0: zero-shot

In [ ]:
# E0: model gốc, NF4 4-bit, KHÔNG huấn luyện — mốc so sánh cho E1–E3.
# Cache-aware: có đủ dòng trong work/e0.jsonl thì bỏ qua, để cell chấm điểm luôn có file đầu vào.
import subprocess, sys
from pathlib import Path

_expected = len([line for line in Path('data/splits/test.jsonl').read_text().splitlines() if line.strip()])
_cached = Path('work/e0.jsonl')
_have = _cached.exists() and len([line for line in _cached.read_text().splitlines() if line.strip()]) == _expected
print('số ảnh test:', _expected, '| work/e0.jsonl đã đủ:', _have)
if not _have:
    subprocess.run([sys.executable, 'student_inference.py', '--split', 'test',
                    '--output', 'work/e0.jsonl'], check=True)


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# CP4: ví dụ huấn luyện và mask loss. Chỉ cần processor + config nên bước này rẻ (không nạp trọng số).
import json
from pathlib import Path

from PIL import Image
from transformers import AutoConfig, AutoProcessor

from scripts.labcommon import read_jsonl
from student_labels import training_target
from student_train import PAD_KEYS, encode_example, make_collate

config = AutoConfig.from_pretrained(MODEL_ID)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'left'

rows = read_jsonl('data/splits/train.jsonl')
example = next(r for r in rows if len(r['objects']) >= 3)
target = training_target(example)
with Image.open(Path('data') / example['image_path']) as handle:
    encoded = encode_example(processor, handle.convert('RGB'), target['answer'])

ids = encoded['input_ids'][0].tolist()
labels = encoded['labels'][0].tolist()
supervised = [t for t in labels if t != -100]
masked = len(labels) - len(supervised)
decoded = processor.tokenizer.decode(supervised)

print('image_id:', example['image_id'])
print('sequence tokens:', len(ids), '| masked (prompt + image):', masked, '| supervised:', len(supervised))
print('vision tokens:', ids.count(config.image_token_id), '(kỳ vọng 220 cho ảnh 640x360)')
print('decoded supervised text:', decoded.replace(chr(10), ' ')[:160])
print('removed from training answer (chính sách đã chốt):', len(target['removed_annotations']))

assert ids.count(config.image_token_id) == 220, 'Sai số vision token: kiểm tra ảnh 640x360 và processor'
assert json.loads(decoded.replace('<|im_end|>', '').strip()) == json.loads(target['answer']), 'Phần được học không khớp JSON đáp án'
assert all(t == -100 for t in labels[:masked]), 'Prompt và token ảnh phải bị mask'
print('CP4 OK: chỉ học phần trả lời, số vision token đúng')

samples = []
for row in rows[:4]:
    with Image.open(Path('data') / row['image_path']) as handle:
        samples.append(encode_example(processor, handle.convert('RGB'),
                                      training_target(row)['answer']))
batch = make_collate(processor.tokenizer.pad_token_id)(samples)
print('collate keys:', sorted(batch))
print('input_ids shape:', tuple(batch['input_ids'].shape),
      '| pixel_values shape:', tuple(batch['pixel_values'].shape))
assert tuple(batch['input_ids'].shape)[0] == len(samples), 'batch phải có đủ số mẫu'
# pixel_values gộp patch của CẢ batch, nên so với tổng patch trong image_grid_thw, không so với số ảnh.
grids = batch['image_grid_thw']
assert tuple(grids.shape)[0] == len(samples), 'mỗi ảnh phải có một dòng grid'
patches = int((grids[:, 0] * grids[:, 1] * grids[:, 2]).sum())
assert tuple(batch['pixel_values'].shape)[0] == patches, \
    f"pixel_values={tuple(batch['pixel_values'].shape)[0]} != tổng patch={patches}"
print('image_grid_thw shape:', tuple(grids.shape), '| tổng patch:', patches)
print('pad keys:', PAD_KEYS)


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# CP5: đối chiếu số module LoRA và số tham số huấn luyện với GUIDE TRƯỚC khi train.
import gc, json, yaml, torch
from pathlib import Path

from student_inference import load_base_model
from student_train import apply_setup

recon = {}
for name in ('e1', 'e2', 'e3'):
    setup = yaml.safe_load(Path(f'configs/{name}.yaml').read_text())
    model, _ = load_base_model(MODEL_ID)
    _, info = apply_setup(model, setup)
    recon[setup['name']] = info
    print(setup['name'], '| targets:', setup['lora']['targets'],
          '| LoRA modules:', info['lora_target_modules'],
          '| projector tensors:', info['projector_tensors_unfrozen'],
          '| trainable params:', f"{info['trainable_params']:,}")
    del model
    gc.collect()
    torch.cuda.empty_cache()

assert recon['E1']['lora_target_modules'] == 252, 'E1 phải có 252 module LoRA (LLM)'
assert recon['E3']['lora_target_modules'] == 348, 'E3 phải có 348 module LoRA (LLM + ViT)'
assert recon['E1']['projector_tensors_unfrozen'] == 0 and recon['E2']['projector_tensors_unfrozen'] > 0
delta = recon['E2']['trainable_params'] - recon['E1']['trainable_params']
print('E2 - E1 trainable params:', f'{delta:,}', '(GUIDE: khoảng 109M)')
assert 0.7e8 < delta < 1.5e8, 'Chênh lệch E2-E1 không khớp mức ~109M của GUIDE'
Path('report/checkpoints').mkdir(parents=True, exist_ok=True)
Path('report/checkpoints/05-lora-recon.json').write_text(json.dumps(recon, indent=2), encoding='utf-8')
print('CP5 OK: cấu hình khớp GUIDE')


## 6. Vòng huấn luyện

In [ ]:
# Vòng huấn luyện dùng chung. Cache-aware: có done.json thì bỏ qua; luôn nạp lại adapter + projector.
import gc, json, subprocess, sys
from pathlib import Path

import torch, yaml

from scripts.labcommon import read_jsonl
from student_inference import generate_predictions, load_base_model
from student_train import load_adapter_and_projector, train_setup


def run_experiment(config_name, fraction=1.0, coord='norm1000', tag=None, evidence_name=None):
    setup = yaml.safe_load(Path(f'configs/{config_name}.yaml').read_text())
    tag = tag or (setup['name'] if fraction == 1.0 else f"{setup['name']}_d{int(fraction * 100)}")
    out = Path('work') / tag.lower()
    out.mkdir(parents=True, exist_ok=True)
    predictions = Path('work') / f'{tag.lower()}.jsonl'
    evidence = Path('report/evidence') / f'{evidence_name or tag.lower()}.json'

    if not (out / 'done.json').exists():
        rows = read_jsonl('data/splits/train.jsonl')
        model, processor = load_base_model(MODEL_ID)
        peft_model, info = train_setup(model, processor, setup, rows, 'data', out, fraction=fraction)
        del peft_model, model, processor
        gc.collect()
        torch.cuda.empty_cache()
    info = json.loads((out / 'run.json').read_text())
    print(tag, '| trainable_params:', f"{info['trainable_params']:,}",
          '| peak_vram_gb:', round(info['peak_vram_gb'], 3),
          '| train_minutes:', round(info['train_minutes'], 2))

    if not predictions.exists():
        base, processor = load_base_model(MODEL_ID)
        infer_model, reloaded = load_adapter_and_projector(base, out)
        print('projector tensors nạp lại:', reloaded)
        stats = generate_predictions(infer_model, processor, read_jsonl('data/splits/test.jsonl'),
                                     'data', predictions, batch_size=4, max_new_tokens=256)
        print('inference:', stats)
        del infer_model, base, processor
        gc.collect()
        torch.cuda.empty_cache()

    subprocess.run([sys.executable, 'scripts/score.py', '--pred', str(predictions),
                    '--name', tag, '--out', str(evidence), '--coord', coord,
                    '--meta', f"trainable_params={info['trainable_params']}",
                    '--meta', f"peak_vram_gb={info['peak_vram_gb']}",
                    '--meta', f"train_minutes={info['train_minutes']}"], check=True)
    return json.loads(evidence.read_text())


print('run_experiment() sẵn sàng. Bước tiếp theo chạy E1, E2, E3.')


## 7. Suy luận và chấm E1–E3

In [ ]:
# Huấn luyện + suy luận + chấm E1, E2, E3. Mỗi setup khoảng 1 giờ trên T4.
results = {}
for name in ('e1', 'e2', 'e3'):
    results[name.upper()] = run_experiment(name)
    entry = results[name.upper()]
    print(name.upper(), '| F1:', round(entry['micro']['f1'], 4),
          '| CI95:', [round(v, 4) for v in entry['f1_ci95']],
          '| valid:', round(entry['parse_valid_rate'], 4),
          '| FP/img:', round(entry['fp_per_image'], 3),
          '| F1 theo lớp:', {k: round(v['f1'], 4) for k, v in entry['per_class'].items()})
print('Xong E1–E3; bằng chứng nằm trong report/evidence/')


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 (ablation): đổi ĐÚNG MỘT biến so với setup tốt nhất của E1–E3 -> chọn data_size.
# Giá trị 100% chính là setup gốc đã chạy, nên chỉ cần train thêm mức 10% (GUIDE cho phép).
import json, subprocess, sys
from pathlib import Path

best = max(results, key=lambda key: results[key]['micro']['f1'])
print('setup tốt nhất theo F1:', best, '->', round(results[best]['micro']['f1'], 4))

ten_percent = run_experiment(best.lower(), fraction=0.10,
                             tag=f'{best}_d10', evidence_name='e4_data_size_0.1')
# Chấm lại chính dự đoán của setup 100% dưới tên E4 để có đủ hai giá trị của biến.
full_predictions = Path('work') / f'{best.lower()}.jsonl'
full_run = json.loads((Path('work') / best.lower() / 'run.json').read_text())
subprocess.run([sys.executable, 'scripts/score.py', '--pred', str(full_predictions),
                '--name', 'E4_data_size_1.0', '--out', 'report/evidence/e4_data_size_1.0.json',
                '--meta', f"trainable_params={full_run['trainable_params']}",
                '--meta', f"peak_vram_gb={full_run['peak_vram_gb']}",
                '--meta', f"train_minutes={full_run['train_minutes']}"], check=True)
full = json.loads(Path('report/evidence/e4_data_size_1.0.json').read_text())

comparison = {
    'changed_variable': 'data_size (train_fraction)',
    'best_setup_100_percent': best,
    'note': 'Giá trị 1.0 dùng lại dự đoán của setup tốt nhất; chỉ train thêm mức 0.1.',
    'values': {
        '0.1': {'f1': ten_percent['micro']['f1'], 'ci95': ten_percent['f1_ci95'],
                'parse_valid_rate': ten_percent['parse_valid_rate'],
                'train_minutes': ten_percent['meta'].get('train_minutes')},
        '1.0': {'f1': full['micro']['f1'], 'ci95': full['f1_ci95'],
                'parse_valid_rate': full['parse_valid_rate'],
                'train_minutes': full['meta'].get('train_minutes')},
    },
}
Path('report/checkpoints').mkdir(parents=True, exist_ok=True)
Path('report/checkpoints/08-e4-data-size.json').write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(comparison, indent=2, ensure_ascii=False))
print('Kết luận phải đọc kèm CI95 — test chỉ có 311 ảnh.')
